# NeuroScan Phase 2 — Track 1 (Kainat)

**Baseline CNN + the data-leakage experiment.**

Phase 1 found that the raw dataset contains 187 exact duplicates, 90 near-duplicates and
**161 training images that also appear in the test set**. The original Phase 2 notebook
trained straight off the raw directories, so it never benefited from any of that cleaning.

This notebook does two things:

1. **A from-scratch CNN baseline** on the clean pipeline — the reference point that says how
   much transfer learning is actually buying us (Laiba's and Mania's tracks).
2. **The leakage experiment** — train the *same* EfficientNetB0, with the *same*
   hyperparameters, once on the clean data and once on the old leaky pipeline, and measure
   the gap. That number is what Phase 1 was for.

> **Run this on a GPU.** Colab: *Runtime → Change runtime type → T4 GPU*. On CPU the
> EfficientNet runs take hours.

Everything shared lives in `neuroscan/` — see `PHASE2.md` for the contract.

## 1. Setup

In [ ]:
# Colab: clone the repo. Local: this cell is a no-op.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO = "NeuroScan"

if IN_COLAB and not os.path.isdir(REPO):
    # Use your own fork/remote here if you are not pushing to this one.
    subprocess.run(["git", "clone", "https://github.com/laibaamjad50/NeuroScan.git"], check=True)
if IN_COLAB:
    os.chdir(REPO)

print("working directory:", os.getcwd())
assert os.path.isdir("neuroscan"), "run this from the repository root"

In [ ]:
!pip install -q -r requirements-phase2.txt

In [ ]:
# processed_data/ is gitignored, so rebuild it from the manifest (a few minutes).
# Reproduces the Phase 1 output exactly; skips anything already on disk.
!python scripts/build_processed_data.py

In [ ]:
import json, os, sys
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, applications

sys.path.insert(0, os.getcwd())
from neuroscan import data, evaluate

SEED = 42
tf.keras.utils.set_random_seed(SEED)

EPOCHS_BASELINE = 25
EPOCHS_PHASE_A  = 15     # matches the original Phase 2 notebook
EPOCHS_PHASE_B  = 15

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE - this will be slow")

report = data.verify(full=True)
print("\nPhase 1 outputs:", "OK" if report["ok"] else "PROBLEM")
for p in report["problems"]:
    print("  -", p)
print("counts:", report["counts"])
CLASSES = report["classes"]
print("classes:", CLASSES)

## 2. Baseline — a small CNN trained from scratch

No pretrained weights, single greyscale channel, inputs scaled to [0, 1]. This is the
"what do we get without transfer learning?" number. If a 21-million-parameter pretrained
backbone can't beat this by a clear margin, that's worth knowing.

In [ ]:
def build_baseline_cnn(input_shape, n_classes=4):
    """Four conv blocks -> global average pool -> dense head."""
    return models.Sequential([
        layers.Input(shape=input_shape),
        *[l for f in (32, 64, 128, 256) for l in (
            layers.Conv2D(f, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(f, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.MaxPooling2D(),
        )],
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation="softmax"),
    ], name="baseline_cnn")


# 1 channel + [0,1] scaling: appropriate for a network trained from scratch.
cfg_base = data.DataConfig(preprocess="rescale", channels=1, batch_size=32, seed=SEED)
ds_base = data.make_datasets(cfg_base)
info = ds_base["info"]
print(info["counts"], "| input", info["input_shape"])

baseline = build_baseline_cnn(info["input_shape"], len(CLASSES))
baseline.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
baseline.summary()

In [ ]:
cb = [
    callbacks.ModelCheckpoint("models/baseline_cnn.keras", monitor="val_accuracy",
                              save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=6,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

hist_base = baseline.fit(
    ds_base["train"],
    validation_data=ds_base["val"],
    epochs=EPOCHS_BASELINE,
    class_weight=info["class_weights"],
    callbacks=cb,
)

In [ ]:
res_base = evaluate.evaluate_model(baseline, ds_base["test"], CLASSES,
                                   name="baseline_cnn", outdir="results")
evaluate.plot_history(hist_base, "results/baseline_cnn_history.png", "baseline CNN")

print(open("results/baseline_cnn_report.txt").read())

## 3. The leakage experiment

The design: **one model, one set of hyperparameters, two datasets.**

| run | training data | test data |
|---|---|---|
| `effnet_clean` | Phase 1 clean split (4,156) | clean test (1,567) |
| `effnet_leaky` | raw `dataset/Training` with a random 20% val split | raw `dataset/Testing` (1,600) |

The leaky run reproduces exactly what the original Phase 2 notebook did. Its training set
still contains the 161 images that also sit in its test set, so it can score well by
recognising images it has memorised.

We also evaluate the **leaky-trained model on the clean test set**. That separates two
different effects: how much came from training on contaminated data, versus how much came
from the test set itself containing duplicates.

Hyperparameters below are copied from the original notebook so the comparison is about the
data and nothing else.

In [ ]:
def build_effnet(input_shape, n_classes=4):
    """EfficientNetB0 + the same head as the original Phase 2 notebook."""
    base = applications.EfficientNetB0(
        input_shape=input_shape, include_top=False, weights="imagenet")
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(n_classes, activation="softmax"),
    ], name="effnet")
    return model, base


def train_effnet(ds, tag, class_weight=None):
    """Two-phase training: frozen base, then fine-tune at a low learning rate."""
    tf.keras.utils.set_random_seed(SEED)
    model, base = build_effnet(ds["info"]["input_shape"], len(CLASSES))
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

    cb = [
        callbacks.ModelCheckpoint(f"models/{tag}.keras", monitor="val_accuracy",
                                  save_best_only=True, verbose=0),
        callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                restore_best_weights=True, verbose=1),
    ]

    print(f"\n=== {tag} | phase A (base frozen) ===")
    h_a = model.fit(ds["train"], validation_data=ds["val"], epochs=EPOCHS_PHASE_A,
                    class_weight=class_weight, callbacks=cb)

    print(f"\n=== {tag} | phase B (fine-tuning) ===")
    base.trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    h_b = model.fit(ds["train"], validation_data=ds["val"], epochs=EPOCHS_PHASE_B,
                    class_weight=class_weight, callbacks=cb)

    evaluate.plot_history([h_a, h_b], f"results/{tag}_history.png", tag)
    return model, [h_a, h_b]

In [ ]:
# --- Run 1: the clean pipeline -------------------------------------------
cfg_eff = data.DataConfig(preprocess="efficientnet", channels=3, batch_size=32, seed=SEED)
ds_clean = data.make_datasets(cfg_eff)

eff_clean, hist_clean = train_effnet(ds_clean, "effnet_clean",
                                     class_weight=ds_clean["info"]["class_weights"])
res_clean = evaluate.evaluate_model(eff_clean, ds_clean["test"], CLASSES,
                                    name="effnet_clean", outdir="results")
print(f"clean  -> accuracy {res_clean['accuracy']:.4f} | macro-F1 {res_clean['macro_f1']:.4f}")

In [ ]:
# --- Run 2: the old leaky pipeline ---------------------------------------
# Same model, same hyperparameters, contaminated data. No class weights here,
# because the original notebook did not use any.
ds_leaky = data.make_leaky_datasets(cfg_eff)
print("leaky pipeline:", ds_leaky["info"]["counts"])

eff_leaky, hist_leaky = train_effnet(ds_leaky, "effnet_leaky", class_weight=None)
res_leaky = evaluate.evaluate_model(eff_leaky, ds_leaky["test"], CLASSES,
                                    name="effnet_leaky", outdir="results")
print(f"leaky  -> accuracy {res_leaky['accuracy']:.4f} | macro-F1 {res_leaky['macro_f1']:.4f}")

In [ ]:
# --- Run 2b: the leaky-trained model, scored on the CLEAN test set --------
# Isolates "trained on contaminated data" from "tested on a contaminated test set".
res_cross = evaluate.evaluate_model(eff_leaky, ds_clean["test"], CLASSES,
                                    name="effnet_leaky_on_clean_test", outdir="results")
print(f"leaky model / clean test -> accuracy {res_cross['accuracy']:.4f} "
      f"| macro-F1 {res_cross['macro_f1']:.4f}")

## 4. Results

In [ ]:
gap_acc = res_leaky["accuracy"] - res_clean["accuracy"]
gap_f1  = res_leaky["macro_f1"] - res_clean["macro_f1"]

print(f"{'run':<34}{'accuracy':>10}{'macro-F1':>10}")
for label, r in (("clean pipeline (honest)", res_clean),
                 ("leaky pipeline (old notebook)", res_leaky),
                 ("leaky model on clean test", res_cross)):
    print(f"{label:<34}{r['accuracy']:>10.4f}{r['macro_f1']:>10.4f}")

print(f"\nInflation from leakage: {gap_acc:+.4f} accuracy, {gap_f1:+.4f} macro-F1")
print(f"i.e. the uncleaned pipeline reports about {abs(gap_acc) * 100:.2f} "
      f"percentage points more accuracy than it has earned.")

summary = {
    "clean": {k: res_clean[k] for k in ("accuracy", "macro_f1")},
    "leaky": {k: res_leaky[k] for k in ("accuracy", "macro_f1")},
    "leaky_on_clean_test": {k: res_cross[k] for k in ("accuracy", "macro_f1")},
    "inflation_accuracy": gap_acc,
    "inflation_macro_f1": gap_f1,
}
with open("results/leakage_experiment.json", "w") as fh:
    json.dump(summary, fh, indent=2)

evaluate.comparison_table("results")

## 5. What to write up

Fill these in once the runs finish:

- **Baseline vs transfer learning** — how much does the pretrained backbone actually add over
  a from-scratch CNN?
- **The inflation number** — the headline Phase 1 result. The uncleaned pipeline reports
  *X* percentage points more accuracy than the clean one. That is the cost of not cleaning,
  and the justification for all of Phase 1.
- **Where the inflation comes from** — compare `effnet_leaky` against
  `effnet_leaky_on_clean_test`. If the leaky model drops sharply on the clean test set, it
  was memorising; if it holds up, the inflation came mostly from duplicates inside the test
  set itself.
- **Per-class recall** — check `results/*_report.txt` for tumours predicted as "no tumour".
  Clinically that's the expensive error, and plain accuracy hides it.

Hand `models/effnet_clean.keras` to Mania for the Grad-CAM work (Track 3). Phase 1 flagged
that resolution correlates with class (Cramér's V = 0.52), so it is worth confirming the
model attends to the tumour rather than to image dimensions.